# Human-in-the-Loop: Porți înainte de acțiune, clasificarea riscurilor și jurnalul de audit

README-ul pentru această lecție introduce Human-in-the-Loop cu un scurt fragment care cere utilizatorului să `APROBE` sau să `RESPINGĂ` după ce agentul a produs deja un răspuns. Acest tipar este un punct de plecare bun, dar implementările HITL de producție necesită în mod obișnuit încă trei elemente suplimentare:

1. O **poartă înainte de acțiune** care funcționează **înainte** ca agentul să execute un pas cu risc, astfel încât costul, ireversibilitatea și latența să rămână sub control.
2. **Clasificarea riscurilor**, astfel încât acțiunile cu risc scăzut se execută automat, cele cu risc mediu sunt aprobate în lot, iar doar acțiunile cu risc ridicat sunt blocate pentru intervenția umană.
3. Un **jurnal de audit plus un ciclu de revizuire**, astfel încât fiecare decizie privind poarta să fie înregistrată ca JSONL, iar o respingere să recheme agentul cu un motiv structurat în loc să afișeze doar `Revizuind...`.

Acest notebook construiește fiecare dintre acestea pe baza acelorași primitive ca în `06-system-message-framework.ipynb`. Rulează complet în `DEMO_MODE = True` (fără necesitatea unei intrări interactive) sau cu prompturi reale `input()` când `DEMO_MODE = False`. Notă: în modul DEMO, reîncercarea scopului al treilea este scriptată astfel încât mecanica ciclului să fie vizibilă complet. Reclasificarea reală bazată pe revizuire necesită `DEMO_MODE = False` și un operator.

**În afara domeniului (gestionat în alte lecții):** autentificarea și controlul accesului (amenințarea 2 din README-ul lecției 06), middleware pentru apeluri de instrumente (explorare aprofundată în lecția 14 MAF), tipare de dezbatere multi-agent.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## Tipar 1: Poartă pre-acțiune

Fragmentul HITL din README apelează mai întâi agentul, apoi îi cere utilizatorului să aprobe rezultatul. Aceasta este o flux **post-acțiune**. Agentul a executat deja, deci costul apelului LLM este deja plătit și orice efect secundar (email trimis, rând scris în baza de date, comentariu postat) s-a produs deja.

Un flux **pre-acțiune** introduce poarta înainte ca agentul să execute pasul cu risc. Agentul propune acțiunea, poarta decide dacă se execută, iar efectul secundar apare doar după aprobare.

| Aspect | Aprobarea post-acțiune (fragment README) | Poartă pre-acțiune (acest caiet) |
|---|---|---|
| Când rulează aprobarea? | După ce agentul a produs rezultatul | Înainte ca vreun efect secundar să se execute |
| Cost LLM la respingere | Deja plătit | Plătit doar pentru propunere, nu pentru acțiune |
| Efecte secundare ireversibile | Posibile (acțiunea s-a produs deja) | Prevenite |
| Claritatea auditului | Aprobarea este o instrucțiune de printare | Aprobarea este un înregistrare JSON cu dată și oră, acțiune, motiv |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Tiparul 2: Clasificarea pe niveluri de risc

Nu orice acțiune necesită aprobare umană. O interogare doar în citire la o API publică are mize diferite față de trimiterea unui email unui client. Tratarea ambelor la fel irosește atenția operatorului și încetinește agentul.

Un model simplu cu 3 niveluri:

| Nivel | Exemple | Flux de aprobare |
|---|---|---|
| `scăzut` (numai citire) | Caută într-o bază de cunoștințe, interoghează opțiuni de zbor, preia o pagină web publică | Executare automată, înregistrată pentru audit |
| `mediu` (modificare ieftină) | Cache-uiește un rezultat, incrementează un contor, programează un memento | Executare automată, dar revizuire zilnică în loturi |
| `ridicat` (cu impact extern sau ireversibil) | Trimite un email, încarcă un card, postează pe un canal public | Blocat până la aprobare umană |

Aceasta este o clasificare pe niveluri. Sistemele de producție folosesc adesea niveluri mai granulare (de ex., niveluri de permisiuni AWS IAM, clasificări bazate pe roluri). Versiunea cu 3 niveluri de mai jos este cea mai mică versiune utilă pentru un agent care combină acțiuni numai în citire și cu efecte secundare.

Clasificatorul de mai jos folosește euristici bazate pe cuvinte cheie pentru ca demo-ul să rămână determinist și ieftin. Într-un sistem de producție l-ai înlocui cu un clasificator învățat sau un motor de politici.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Modelul 3: Jurnal de audit și buclă de revizie

Un `print("Response approved.")` nu este un jurnal de audit. Pentru încredere, fiecare decizie a porții ar trebui înregistrată ca un eveniment structurat pe care îl puteți interoga, reda sau atașa ulterior la o revizuire a incidentului.

Două componente:

1. **JSONL doar cu adăugări.** O linie per decizie, cu marcaj temporal, acțiune, nivel, decizie, motiv. Ușor de căutat, ușor de trimis ulterior către un depozit real de jurnale.
2. **Bucla de revizie la respingere.** Când poarta întoarce `deny`, agentul își reintroduce solicitarea cu motivul respingerii în context, astfel încât următoarea propunere să poată evita problema.


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Resurse suplimentare

Mai multe alte proiecte publice implementează variații ale acestor modele HITL. Compară abordările pentru a găsi ce se potrivește stivei tale:

- **LangChain** instrumente human-in-the-loop învelitoare ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): învelitori de instrumente plug-and-play care opresc execuția pentru input uman.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ a restructurat asta): folosește un rol de agent specific pentru a reprezenta omul în conversații multi-agent.
- **Microsoft Agent Framework (MAF)** middleware pentru invocarea funcțiilor ([docs](https://learn.microsoft.com/agent-framework/)): middleware care rulează în jurul fiecărei apeluri de instrument/functie, potrivit pentru logica de restricționare și fluxurile de aprobare.

Fiecare proiect gestionează cele trei sub-modele diferit: LangChain le învelește ca instrumente, AutoGen folosește un rol de agent, iar Microsoft Agent Framework folosește middleware pentru invocarea funcțiilor. Citește una sau două implementări complet înainte de a alege un design pentru propriul tău agent.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Declinare a responsabilității**:
Acest document a fost tradus folosind serviciul de traducere AI [Co-op Translator](https://github.com/Azure/co-op-translator). În timp ce ne străduim pentru acuratețe, vă rugăm să rețineți că traducerile automate pot conține erori sau inexactități. Documentul original în limba sa nativă trebuie considerat sursa autorizată. Pentru informații critice, se recomandă traducerea profesională realizată de un om. Nu ne asumăm responsabilitatea pentru eventualele neînțelegeri sau interpretări greșite care decurg din utilizarea acestei traduceri.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
